In [ ]:
# %% [markdown]
# Pew-based User Simulation (Listeners)
# Author: <your name>
# Source: Pew Research Center (Dec 5-11, 2022) "Podcasts as a Source of News and Information"
# Notes:
# - Topic (12 subject) and Listening Frequency targets are taken directly from Pew (% among listeners).
# - Demographics (Age, Gender, Education, Income) use Pew's "% of group who listened in past 12 months"
#   as weights to form a listener sample; these are NOT population marginals but listener-propensity weights.
# - We normalize each target vector to sum to 1 for single-label draws.
# - Outputs: target vs simulated comparison tables + distribution tables, saved to CSV.

# %%capture
!pip install pandas numpy pyarrow --quiet

import pandas as pd
import numpy as np

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# -----------------------------
# 1) PEW TARGETS (from the report)
# -----------------------------
# 1A) Topics regularly listened to (12 subjects) — % among podcast listeners
# Ref (report tables/figures): Comedy 47, Entertainment/Pop/Arts 46, Politics&Gov 41, Sci&Tech 40, History 40,
# True Crime 34, Self-Help&Relationships 32, Money&Finance 31, Religion&Spirituality 30, Health&Fitness 27,
# Sports 22, Race&Ethnicity 15.
topic_targets_pct = {
    "Comedy": 47,
    "Entertainment / Pop Culture / Arts": 46,
    "Politics & Government": 41,
    "Science & Technology": 40,
    "History": 40,
    "True Crime": 34,
    "Self-Help & Relationships": 32,
    "Money & Finance": 31,
    "Religion & Spirituality": 30,
    "Health & Fitness": 27,
    "Sports": 22,
    "Race & Ethnicity": 15,
}

# 1B) Listening frequency — % among podcast listeners
# Nearly every day 20, Few times a week 22, Few times a month 25, Once a month 9, Less often 25.
freq_targets_pct = {
    "Nearly every day": 20,
    "A few times a week": 22,
    "A few times a month": 25,
    "Once a month": 9,
    "Less often": 25,
}

# 1C) Demographic listener-propensity weights
# These are "% of the group who listened in the past 12 months" (NOT population shares).
# We will normalize them to use as sampling weights to build a listener sample.

# Age groups (report Appendix): 18-29:67, 30-49:58, 50-64:42, 65+:28
age_listen_pct = {
    "18–29": 67,
    "30–49": 58,
    "50–64": 42,
    "65+": 28,
}

# Gender: Men 51, Women 46  (share who listened in past 12 months)
gender_listen_pct = {
    "Male": 51,
    "Female": 46,
}

# Education: HS or less 37, Some college 49, College+ 62
edu_listen_pct = {
    "High school or less": 37,
    "Some college": 49,
    "College+": 62,
}

# Income: <30K 44, 30K–79,999 45, ≥80K 59
income_listen_pct = {
    "<30K": 44,
    "30K–79,999": 45,
    "80K+": 59,
}

# -----------------------------
# 2) Helper: normalize % -> probabilities
# -----------------------------
def normalize_pct_dict(d):
    vals = np.array(list(d.values()), dtype=float)
    p = vals / vals.sum()
    return dict(zip(d.keys(), p))

topic_prob = normalize_pct_dict(topic_targets_pct)
freq_prob  = normalize_pct_dict(freq_targets_pct)
age_prob   = normalize_pct_dict(age_listen_pct)
gender_prob= normalize_pct_dict(gender_listen_pct)
edu_prob   = normalize_pct_dict(edu_listen_pct)
inc_prob   = normalize_pct_dict(income_listen_pct)

# -----------------------------
# 3) Simulate N listeners
# -----------------------------
N = 2000  # change if needed

def draw_from_prob_map(prob_map, size):
    cats = list(prob_map.keys())
    probs = np.array(list(prob_map.values()), dtype=float)
    return rng.choice(cats, size=size, p=probs)

# Draw demographics using normalized listener-propensity weights
age_draw    = draw_from_prob_map(age_prob,    N)
gender_draw = draw_from_prob_map(gender_prob, N)
edu_draw    = draw_from_prob_map(edu_prob,    N)
inc_draw    = draw_from_prob_map(inc_prob,    N)

# Draw single preferred topic (Pew topics normalized to 1 for single-choice draw)
topic_draw  = draw_from_prob_map(topic_prob,  N)

# Draw listening frequency (among listeners)
freq_draw   = draw_from_prob_map(freq_prob,   N)

users = pd.DataFrame({
    "Age Group": age_draw,
    "Gender": gender_draw,
    "Education": edu_draw,
    "Income": inc_draw,
    "Preferred Category (Pew12)": topic_draw,
    "Listening Frequency": freq_draw,
})

# -----------------------------
# 4) Build TARGET vs SIMULATED comparison tables
# -----------------------------
def target_vs_sim(df, col, targets_pct_dict):
    # empirical share
    emp = (df[col].value_counts(normalize=True) * 100).reindex(targets_pct_dict.keys(), fill_value=0).round(2)
    tgt = pd.Series(targets_pct_dict, dtype=float)
    out = pd.DataFrame({
        "Target (%)": tgt,
        "Simulated (%)": emp
    })
    out["Abs. Diff (pp)"] = (out["Simulated (%)"] - out["Target (%)"]).round(2)
    return out

cmp_topics = target_vs_sim(users, "Preferred Category (Pew12)", topic_targets_pct)
cmp_freq   = target_vs_sim(users, "Listening Frequency", freq_targets_pct)

# For demographics, the "targets" we are matching are listener-propensity weights normalized to 100%.
# Bu tablo "kullanıcı örneklemimizin bu ağırlıkları takip ettiğini" gösterir.
def weights_vs_sim(df, col, weights_pct_dict):
    tgt_norm = normalize_pct_dict(weights_pct_dict)
    tgt_pct  = {k: v*100 for k, v in tgt_norm.items()}
    return target_vs_sim(df, col, tgt_pct)

cmp_age    = weights_vs_sim(users, "Age Group", age_listen_pct)
cmp_gender = weights_vs_sim(users, "Gender", gender_listen_pct)
cmp_edu    = weights_vs_sim(users, "Education", edu_listen_pct)
cmp_income = weights_vs_sim(users, "Income", income_listen_pct)

# -----------------------------
# 5) Distribution tables (value counts) for the generated data
# -----------------------------
def dist_table(df, col):
    vc = df[col].value_counts().rename("Count")
    share = (df[col].value_counts(normalize=True)*100).rename("Share (%)").round(2)
    return pd.concat([vc, share], axis=1)

dist_age    = dist_table(users, "Age Group")
dist_gender = dist_table(users, "Gender")
dist_edu    = dist_table(users, "Education")
dist_income = dist_table(users, "Income")
dist_topic  = dist_table(users, "Preferred Category (Pew12)")
dist_freq   = dist_table(users, "Listening Frequency")

# -----------------------------
# 6) Save outputs
# -----------------------------
users_path_csv = "simulated_users_pew_ALL.csv"
users.to_csv(users_path_csv, index=False)

with pd.ExcelWriter("pew_targets_vs_simulated.xlsx") as xw:
    cmp_topics.to_excel(xw, sheet_name="Topics_Target_vs_Sim", index=True)
    cmp_freq.to_excel(xw,   sheet_name="Freq_Target_vs_Sim", index=True)
    cmp_age.to_excel(xw,    sheet_name="Age_Weights_vs_Sim", index=True)
    cmp_gender.to_excel(xw, sheet_name="Gender_Weights_vs_Sim", index=True)
    cmp_edu.to_excel(xw,    sheet_name="Edu_Weights_vs_Sim", index=True)
    cmp_income.to_excel(xw, sheet_name="Income_Weights_vs_Sim", index=True)

with pd.ExcelWriter("simulated_distributions.xlsx") as xw:
    dist_age.to_excel(xw,    sheet_name="Age", index=True)
    dist_gender.to_excel(xw, sheet_name="Gender", index=True)
    dist_edu.to_excel(xw,    sheet_name="Education", index=True)
    dist_income.to_excel(xw, sheet_name="Income", index=True)
    dist_topic.to_excel(xw,  sheet_name="Topics", index=True)
    dist_freq.to_excel(xw,   sheet_name="Frequency", index=True)

print("Saved:")
print(" - simulated_users_pew_ALL.csv")
print(" - pew_targets_vs_simulated.xlsx")
print(" - simulated_distributions.xlsx")

# Peek at key tables
cmp_topics, cmp_freq, cmp_age, cmp_gender, cmp_edu, cmp_income


In [ ]:
from google.colab import files
files.download("simulated_distributions.xlsx")

In [ ]:
from google.colab import files
files.download("pew_targets_vs_simulated.xlsx")

In [ ]:
from google.colab import files
files.download("simulated_users_pew_ALL.csv")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Topic Distribution
plt.figure(figsize=(10,6))
sns.countplot(y="Preferred Category (Pew12)", data=users,
              order=users["Preferred Category (Pew12)"].value_counts().index,
              palette="viridis")
plt.title("Simulated Users: Preferred Podcast Categories (Pew12)")
plt.xlabel("Count")
plt.ylabel("Category")
plt.show()

# Listening Frequency
plt.figure(figsize=(8,5))
sns.countplot(x="Listening Frequency", data=users,
              order=["Nearly every day","A few times a week","A few times a month","Once a month","Less often"],
              palette="magma")
plt.title("Simulated Users: Listening Frequency")
plt.ylabel("Count")
plt.show()

# Age Distribution
plt.figure(figsize=(6,4))
sns.countplot(x="Age Group", data=users,
              order=["18–29","30–49","50–64","65+"], palette="coolwarm")
plt.title("Simulated Users: Age Groups")
plt.show()


In [ ]:
# %% [markdown]
# Comparative visuals for Pew-based simulated dataset
# Works in Google Colab. Saves PNGs under figures/.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- If targets are not in memory, (re)define them here ----------
# Topics (regular listeners %, Pew)
topic_targets_pct = {
    "Comedy": 47, "Entertainment / Pop Culture / Arts": 46, "Politics & Government": 41,
    "Science & Technology": 40, "History": 40, "True Crime": 34, "Self-Help & Relationships": 32,
    "Money & Finance": 31, "Religion & Spirituality": 30, "Health & Fitness": 27, "Sports": 22, "Race & Ethnicity": 15
}
# Frequency (among listeners %, Pew)
freq_targets_pct = {
    "Nearly every day": 20, "A few times a week": 22, "A few times a month": 25,
    "Once a month": 9, "Less often": 25
}
# Demographic listener-propensity (% of group listened in past 12 months, Pew) — normalized for sampling
age_listen_pct = {"18–29": 67, "30–49": 58, "50–64": 42, "65+": 28}
gender_listen_pct = {"Male": 51, "Female": 46}
edu_listen_pct = {"High school or less": 37, "Some college": 49, "College+": 62}
income_listen_pct = {"<30K": 44, "30K–79,999": 45, "80K+": 59}

# ---------- Helper: normalize to probability weights (for targets rendered as shares) ----------
def pct_to_share(d):
    vals = np.array(list(d.values()), dtype=float)
    s = vals / vals.sum()
    return dict(zip(d.keys(), s))

# ---------- Expect a DataFrame 'users' in memory; if not, load it from CSV ----------
if 'users' not in globals():
    if os.path.exists('simulated_users_pew_ALL.csv'):
        users = pd.read_csv('simulated_users_pew_ALL.csv')
    else:
        raise RuntimeError("users DataFrame not found. Load or create 'users' first.")

# ---------- Ensure output dir ----------
os.makedirs("figures", exist_ok=True)

# ---------- Utility: grouped bar (Target vs Simulated) ----------
def plot_target_vs_sim(df, col, target_pct_dict, order=None, title=None, filename=None, rotate=0):
    # Simulated shares
    sim = (df[col].value_counts(normalize=True)).sort_index()
    # Target -> normalize to shares (when % not summing to 100 or are 'propensities')
    tar_share = pd.Series(pct_to_share(target_pct_dict))
    # Align indices
    if order is None:
        order = list(tar_share.index)
    sim = sim.reindex(order).fillna(0.0)
    tar = tar_share.reindex(order).fillna(0.0)

    x = np.arange(len(order))
    width = 0.38

    plt.figure(figsize=(10, 5))
    plt.bar(x - width/2, (tar*100).values, width, label='Target (%)')
    plt.bar(x + width/2, (sim*100).values, width, label='Simulated (%)')
    plt.xticks(x, order, rotation=rotate, ha='right' if rotate else 'center')
    plt.ylabel('Share (%)')
    plt.title(title or f"{col}: Target vs Simulated")
    plt.legend()
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join("figures", filename), dpi=200)
    plt.show()

# ---------- Utility: stacked and 100% stacked bars ----------
def plot_stacked_bar(df, col, title=None, filename=None, normalize=False, order=None, rotate=0):
    counts = df[col].value_counts().sort_index()
    if order is not None:
        counts = counts.reindex(order).fillna(0)
    if normalize:
        shares = counts / counts.sum()
        vals = shares.values
        ylabel = 'Share (%)'
        height = vals * 100
    else:
        vals = counts.values
        ylabel = 'Count'
        height = vals

    plt.figure(figsize=(9, 4))
    # Single stacked bar (useful for compact overview)
    plt.bar([0], height.sum(), color='lightgray')
    bottom = 0
    colors = plt.cm.tab20(np.linspace(0, 1, len(counts)))
    for i, (cat, val) in enumerate(zip(counts.index, height)):
        plt.bar([0], val, bottom=bottom, label=str(cat), color=colors[i])
        bottom += val
    plt.xticks([0], ["All"])
    plt.ylabel(ylabel)
    ttl = title or f"{col} distribution" + (" (100% stacked)" if normalize else "")
    plt.title(ttl)
    if len(counts) <= 12:
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join("figures", filename), dpi=200)
    plt.show()

# ---------- Utility: heatmap from crosstab ----------
def plot_heatmap_crosstab(df, rows, cols, normalize_rows=False, title=None, filename=None):
    ct = pd.crosstab(df[rows], df[cols])
    if normalize_rows:
        ct = ct.div(ct.sum(axis=1), axis=0) * 100.0  # row-wise %
    plt.figure(figsize=(10, 6))
    im = plt.imshow(ct.values, aspect='auto')
    plt.colorbar(im, fraction=0.046, pad=0.04, label='% of row' if normalize_rows else 'Count')
    plt.xticks(range(ct.shape[1]), ct.columns, rotation=45, ha='right')
    plt.yticks(range(ct.shape[0]), ct.index)
    plt.title(title or f"{rows} × {cols} Heatmap" + (" (% rows)" if normalize_rows else ""))
    # annotate
    for i in range(ct.shape[0]):
        for j in range(ct.shape[1]):
            val = ct.values[i, j]
            txt = f"{val:.1f}%" if normalize_rows else f"{int(val)}"
            plt.text(j, i, txt, ha='center', va='center', fontsize=8)
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join("figures", filename), dpi=220)
    plt.show()

# ---------- 1) Target vs Simulated: Topics ----------
topic_order = list(topic_targets_pct.keys())
plot_target_vs_sim(
    users, "Preferred Category (Pew12)", topic_targets_pct,
    order=topic_order,
    title="Preferred Categories (Pew12): Target vs Simulated",
    filename="target_vs_sim_topics.png",
    rotate=45
)

# ---------- 2) Target vs Simulated: Frequency ----------
freq_order = ["Nearly every day","A few times a week","A few times a month","Once a month","Less often"]
plot_target_vs_sim(
    users, "Listening Frequency", freq_targets_pct,
    order=freq_order,
    title="Listening Frequency: Target vs Simulated",
    filename="target_vs_sim_frequency.png",
    rotate=20
)

# ---------- 3) Target vs Simulated: Demographics (weights normalized to shares) ----------
plot_target_vs_sim(users, "Age Group", age_listen_pct,
                   order=["18–29","30–49","50–64","65+"],
                   title="Age Groups (listener-propensity normalized): Target vs Simulated",
                   filename="target_vs_sim_age.png", rotate=0)

plot_target_vs_sim(users, "Gender", gender_listen_pct,
                   order=["Male","Female"],
                   title="Gender (listener-propensity normalized): Target vs Simulated",
                   filename="target_vs_sim_gender.png", rotate=0)

plot_target_vs_sim(users, "Education", edu_listen_pct,
                   order=["High school or less","Some college","College+"],
                   title="Education (listener-propensity normalized): Target vs Simulated",
                   filename="target_vs_sim_education.png", rotate=0)

plot_target_vs_sim(users, "Income", income_listen_pct,
                   order=["<30K","30K–79,999","80K+"],
                   title="Income (listener-propensity normalized): Target vs Simulated",
                   filename="target_vs_sim_income.png", rotate=0)

# ---------- 4) Stacked bars (distribution snapshots) ----------
plot_stacked_bar(users, "Preferred Category (Pew12)",
                 title="Preferred Categories — 100% Stacked",
                 filename="stacked_topics_100pct.png",
                 normalize=True, order=topic_order, rotate=0)

plot_stacked_bar(users, "Listening Frequency",
                 title="Listening Frequency — 100% Stacked",
                 filename="stacked_frequency_100pct.png",
                 normalize=True, order=freq_order, rotate=0)

# ---------- 5) Heatmaps: cross-tabs to see “kim neyi dinliyor?” ----------
# Age × Topic (row-normalized %)
plot_heatmap_crosstab(users, rows="Age Group", cols="Preferred Category (Pew12)",
                      normalize_rows=True,
                      title="Age × Preferred Category (% within Age Group)",
                      filename="heatmap_age_x_topic_rowpct.png")

# Education × Topic (row-normalized %)
plot_heatmap_crosstab(users, rows="Education", cols="Preferred Category (Pew12)",
                      normalize_rows=True,
                      title="Education × Preferred Category (% within Education)",
                      filename="heatmap_edu_x_topic_rowpct.png")

# Income × Frequency (counts)
plot_heatmap_crosstab(users, rows="Income", cols="Listening Frequency",
                      normalize_rows=False,
                      title="Income × Listening Frequency (Counts)",
                      filename="heatmap_income_x_freq_counts.png")

print("Saved figures in ./figures")
